# AdaFace ONNX Evaluation
This notebook evaluates a pretrained `best_adaface.onnx` model by extracting embeddings and computing cosine similarity on your dataset.

In [22]:
!pip install onnxruntime-gpu opencv-python scikit-learn tqdm matplotlib


In [23]:
import cv2
import numpy as np
import onnxruntime as ort
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm


In [25]:
MODEL_PATH="C:\\Users\\taran\\Documents\\GitHub\\vizage\\backend\\models\\best_adaface.onnx"

session=ort.InferenceSession(
    MODEL_PATH,
    providers=['CUDAExecutionProvider','CPUExecutionProvider']
)

print("Input :",session.get_inputs()[0].name)
print("Shape :",session.get_inputs()[0].shape)
print("Output:",session.get_outputs()[0].name)
print("Shape :",session.get_outputs()[0].shape)


Input : input
Shape : ['batch_size', 3, 112, 112]
Output: output
Shape : ['batch_size', 512]


In [26]:
def preprocess(path,size=(112,112)):
    img=cv2.imread(str(path))
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    img=cv2.resize(img,size)
    img=img.astype(np.float32)
    img=(img/255.0-0.5)/0.5
    img=np.transpose(img,(2,0,1))
    img=np.expand_dims(img,0)
    return img


In [27]:
input_name=session.get_inputs()[0].name
output_name=session.get_outputs()[0].name

def embedding(image_path):
    x=preprocess(image_path)
    emb=session.run([output_name],{input_name:x})[0]
    emb=emb/np.linalg.norm(emb)
    return emb


In [28]:
# Change this path
DATASET=Path("../datasets/CV Dataset/Ankit-CV Dataset")

people={}
for person in DATASET.iterdir():
    if person.is_dir():
        people[person.name]=list(person.rglob("*.jpg"))

print("People:",len(people))
print(next(iter(people.items())))


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\datasets\\CV Dataset\\Ankit-CV Dataset'

In [ ]:
# Example: compare two images of same identity
person=list(people.keys())[0]
imgs=people[person]

emb1=embedding(imgs[0])
emb2=embedding(imgs[1])

score=cosine_similarity(emb1,emb2)[0][0]
print("Identity:",person)
print("Similarity:",score)


In [ ]:
# Compare different identities
names=list(people.keys())

emb1=embedding(people[names[0]][0])
emb2=embedding(people[names[1]][0])

score=cosine_similarity(emb1,emb2)[0][0]

print(names[0],"vs",names[1])
print(score)


## Next Step

If the ONNX model outputs 512-dimensional embeddings and the similarity scores behave as expected (same-identity scores consistently higher than different-identity scores), the model is likely functioning correctly. You can then extend this notebook to evaluate all image pairs and compute ROC, EER, FAR/FRR, and overall verification accuracy.
